In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
file_path = '../Datasets/Per_UE_Datasets_final/ue_8642840401612300_malicious_labeled.csv'
df = pd.read_csv(file_path)

# Convert _time to datetime and sort
df['_time'] = pd.to_datetime(df['_time'])
df = df.sort_values(by=['multiclass_label', '_time'])

# Convert bitrates from bps to Mbps
df['dl_bitrate'] = df['dl_bitrate'] / (1024 * 1024)
df['ul_bitrate'] = df['ul_bitrate'] / (1024 * 1024)

# Select key features
features = ['ul_retx', 'dl_bitrate', 'ul_bitrate']

# Create a uniform time step column from 0 to 99 for the first 100 time steps per class
df['time_step'] = df.groupby('multiclass_label').cumcount()

# Keep only the first 100 time steps per class
df = df[df['time_step'] < 100]

# Convert multiclass_label to string for proper plotting
df['multiclass_label'] = df['multiclass_label'].astype(str)

# Define a mapping for class labels
class_label_mapping = {
    "0": "Normal",
    "1": "SYN",
    "2": "ICMP",
    "3": "UDP",
    "4": "DNS",
    "5": "GTPU"
}

# Define folder to save figures
output_folder = "plots_output_new"
os.makedirs(output_folder, exist_ok=True)  # Create folder if it doesn't exist

# Sort class labels to ensure correct order
class_labels = sorted(df['multiclass_label'].unique(), key=lambda x: int(x))  # Sort numerically

# Define colors for features
feature_colors = {
    "ul_retx": "#1f77b4",   # Blue
    "dl_bitrate": "#d62728",  # Red
    "ul_bitrate": "#2ca02c"  # Green
}

# Determine y-axis limits for each feature to ensure consistency
y_limits = {
    feature: (df[feature].min(), df[feature].max()) for feature in features
}

for class_label in class_labels:
    class_name = class_label_mapping[class_label]  
    class_df = df[df['multiclass_label'] == class_label]  
    
    for feature in features:
        plt.figure(figsize=(4, 4))  
        
        # Plot with thick solid lines
        sns.lineplot(
            data=class_df, 
            x='time_step', 
            y=feature, 
            linewidth=4,  # Extra thick lines
            alpha=1.0,  
            color=feature_colors[feature]  # Solid line, no dashes
        )
        # Adjust y-axis to prevent near-zero values from disappearing
        y_min, y_max = y_limits[feature]
        if y_min < 0.1 * y_max:
            y_min = -0.1 * y_max  # Adds a buffer
        plt.ylim(y_min, y_max)

        plt.xlabel('Time Step', fontsize=24)
        plt.ylabel(f'{feature} (Mbps)' if "bitrate" in feature else feature, fontsize=24)
        plt.xticks(fontsize=24)
        plt.yticks(fontsize=24)

        plt.grid(True, linestyle="--", alpha=0.5)

        # Save as PDF
        file_name = f"{feature}_{class_name}.pdf".replace(" ", "_")  
        plt.savefig(os.path.join(output_folder, file_name), format="pdf", bbox_inches='tight')
        plt.close()

print(f"All figures saved in '{output_folder}' as PDF files.")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load datasets
file_path_UE1 = '../Datasets/Per_UE_Datasets_final/ue_8642840401612300_malicious_labeled.csv'
file_path_UE2 = '../Datasets/Per_UE_Datasets_final/ue_8628490433231158_normal_labeled.csv'
file_path_UE3 = '../Datasets/Per_UE_Datasets_final/ue_8609960480859058_normal_labeled.csv'
file_path_UE4 = '../Datasets/Per_UE_Datasets_final/ue_8609960480666910_normal_labeled.csv'
file_path_UE5 = '../Datasets/Per_UE_Datasets_final/ue_8609960468879057_normal_labeled.csv'

df = pd.read_csv(file_path_UE1)
df_2 = pd.read_csv(file_path_UE2)

# Convert _time to datetime and sort
df['_time'] = pd.to_datetime(df['_time'])
df = df.sort_values(by=['multiclass_label', '_time'])

df_2['_time'] = pd.to_datetime(df_2['_time'])
df_2 = df_2.sort_values(by=['_time'])  # Sort only by time

# Convert bitrates from bps to Mbps
df['dl_bitrate'] = df['dl_bitrate'] / (1024 * 1024)
df['ul_bitrate'] = df['ul_bitrate'] / (1024 * 1024)

df_2['dl_bitrate'] = df_2['dl_bitrate'] / (1024 * 1024)
df_2['ul_bitrate'] = df_2['ul_bitrate'] / (1024 * 1024)

# Select key features
features = ['ul_retx', 'dl_bitrate', 'ul_bitrate']

# Step 1: Get first 100 samples per attack type from df (Malicious UE) and copy it
df_filtered = df.groupby('multiclass_label').head(100).copy()

# Step 2: Assign time_step values
df_filtered['time_step'] = df_filtered.groupby('multiclass_label').cumcount()

# Step 3: Extract timestamps from df_filtered
timestamps = df_filtered['_time'].unique()

# Step 4: Get corresponding samples from df_2 (Normal UE) using timestamps and copy it
df_2_filtered = df_2[df_2['_time'].isin(timestamps)].copy()
df_2_filtered['time_step'] = df_filtered['time_step'].values  # Ensure alignment

# Define class label mapping (Keys as Strings)
class_label_mapping = {
    "0": "Normal",
    "1": "SYN",
    "2": "ICMP",
    "3": "UDP",
    "4": "DNS",
    "5": "GTPU"
}

# Define output folder
output_folder = "plots_output_fixed"
os.makedirs(output_folder, exist_ok=True)

# Sort class labels numerically
class_labels = sorted(df_filtered['multiclass_label'].unique(), key=lambda x: int(x))

# Define fixed colors for both UEs
ue_colors = {
    "UE1": "red",   # Red for Malicious UE
    "UE2": "blue"  # Blue for Normal UE
    
}

# Determine y-axis limits for each feature
y_limits = {
    feature: (
        min(df_filtered[feature].min(), df_2_filtered[feature].min()),  # Min from both UEs
        max(df_filtered[feature].max(), df_2_filtered[feature].max())   # Max from both UEs
    ) for feature in features
}

# Loop through each attack type (class label)
for class_label in class_labels:
    class_name = class_label_mapping[str(class_label)]  # Fix KeyError

    # Filter UE1 dataset for the given class
    class_df_ue1 = df_filtered[df_filtered['multiclass_label'] == class_label]

    # Filter UE2 dataset using timestamps instead of indices
    class_df_ue2 = df_2_filtered[df_2_filtered['_time'].isin(class_df_ue1['_time'])]

    for feature in features:
        plt.figure(figsize=(4, 4))  # Square figure for clarity

        # Plot UE1 (Malicious UE) - Solid Line
        sns.lineplot(
            data=class_df_ue1,
            x='time_step',
            y=feature,
            linewidth=4,
            alpha=0.95,
            color=ue_colors["UE1"],
            label="UE1 (Malicious)"
        )

        # Plot UE2 (Normal UE) - Dashed Line
        sns.lineplot(
            data=class_df_ue2,
            x='time_step',
            y=feature,
            linewidth=4,
            alpha=0.95,
            color=ue_colors["UE2"],
            label="UE2 (Normal)"
        )

#         # Adjust y-axis to prevent near-zero values from disappearing
#         y_min, y_max = y_limits[feature]
#         if y_min < 0.1 * y_max:
#             y_min = -0.1 * y_max  # Adds a buffer
#         plt.ylim(y_min, y_max)

        # Adjust y-axis to prevent near-zero values from disappearing
        y_min, y_max = y_limits[feature]

        # Adjust y_min to ensure small values don't disappear
        if y_min < 0.1 * y_max:
            y_min = -0.1 * y_max  # Adds a buffer

        # Adjust y_max to provide extra space and avoid cutting off peaks
        y_max = 1.1 * abs(y_max)  # Increase max limit by 10%

        plt.ylim(y_min, y_max)  # Apply the updated y-axis limits
        
        # Adjust y-axis for fair comparison
        #plt.ylim(y_limits[feature])

        # Improve readability
        plt.xlabel('Time Step', fontsize=24)
        plt.ylabel(f'{feature} (Mbps)' if "bitrate" in feature else feature, fontsize=24)
        plt.xticks(fontsize=24)
        plt.yticks(fontsize=24)
        #plt.legend(fontsize=18)
        
        # Remove legend from the figure
        plt.legend().set_visible(False)  

        plt.grid(True, linestyle="--", alpha=0.5)

        # Save as PDF
        file_name = f"{feature}_{class_name}_comparison.pdf".replace(" ", "_")
        plt.savefig(os.path.join(output_folder, file_name), format="pdf", bbox_inches='tight', dpi=400)
        plt.close()

print(f"✅ All comparison figures saved in '{output_folder}' as high-quality PDFs.")


In [ ]:
# Define file paths
file_paths = {
'UE1': '../Datasets/Per_UE_Datasets_final/ue_8642840401612300_malicious_labeled.csv',
'UE2': '../Datasets/Per_UE_Datasets_final/ue_8628490433231158_normal_labeled.csv',
'UE3': '../Datasets/Per_UE_Datasets_final/ue_8609960480859058_normal_labeled.csv',
'UE4': '../Datasets/Per_UE_Datasets_final/ue_8609960480666910_normal_labeled.csv',
'UE5': '../Datasets/Per_UE_Datasets_final/ue_8609960468879057_normal_labeled.csv'
}

# Load datasets into DataFrames
df_ue1 = pd.read_csv(file_paths["UE1"])
df_ue2 = pd.read_csv(file_paths["UE2"])
df_ue3 = pd.read_csv(file_paths["UE3"])
df_ue4 = pd.read_csv(file_paths["UE4"])
df_ue5 = pd.read_csv(file_paths["UE5"])

# Convert _time to datetime and sort
df_ue1['_time'] = pd.to_datetime(df_ue1['_time'])
df_ue1 = df_ue1.sort_values(by=['multiclass_label', '_time'])

for df in [df_ue2, df_ue3, df_ue4, df_ue5]:
    df['_time'] = pd.to_datetime(df['_time'])
    df.sort_values(by=['_time'], inplace=True)

# Convert bitrates from bps to Mbps
for df in [df_ue1, df_ue2, df_ue3, df_ue4, df_ue5]:
    df['ul_bitrate'] = df['ul_bitrate'] / (1024 * 1024)

# Select key features
features = ['ul_retx', 'ul_bitrate']

# Step 1: Get first 100 samples per attack type from Malicious UE (UE1)
df_filtered = df_ue1.groupby('multiclass_label').head(100).copy()

# Step 2: Assign time_step values
df_filtered['time_step'] = df_filtered.groupby('multiclass_label').cumcount()

# Step 3: Extract timestamps from df_filtered
timestamps = df_filtered['_time'].unique()

# Step 4: Get corresponding samples from Normal UEs (UE2, UE3, UE4, UE5) and compute the average
df_ue2_filtered = df_ue2[df_ue2['_time'].isin(timestamps)].copy()
df_ue3_filtered = df_ue3[df_ue3['_time'].isin(timestamps)].copy()
df_ue4_filtered = df_ue4[df_ue4['_time'].isin(timestamps)].copy()
df_ue5_filtered = df_ue5[df_ue5['_time'].isin(timestamps)].copy()

# Convert bitrates from bps to Mbps
for df in [df_filtered, df_ue2_filtered, df_ue3_filtered, df_ue4_filtered, df_ue5_filtered]:
    print(df)
    
# Ensure time alignment
df_ue2_filtered['time_step'] = df_filtered['time_step'].values
df_ue3_filtered['time_step'] = df_filtered['time_step'].values
df_ue4_filtered['time_step'] = df_filtered['time_step'].values
df_ue5_filtered['time_step'] = df_filtered['time_step'].values

# Compute the average feature values for normal UEs
df_normal_avg = df_ue2_filtered.copy()
for feature in features:
    df_normal_avg[feature] = (df_ue2_filtered[feature] + df_ue3_filtered[feature] +
                              df_ue4_filtered[feature] + df_ue5_filtered[feature]) / 4

# Define class label mapping
class_label_mapping = {
    "0": "Normal",
    "1": "SYN",
    "2": "ICMP",
    "3": "UDP",
    "4": "DNS",
    "5": "GTPU"
}

# Define output folder
output_folder = "plots_output_fixed_average"
os.makedirs(output_folder, exist_ok=True)

# Sort class labels numerically
class_labels = sorted(df_filtered['multiclass_label'].unique(), key=lambda x: int(x))

# Define fixed colors
ue_colors = {
    "Malicious": "red",   # Red for Malicious UE
    "Normal": "blue"      # Blue for Averaged Normal UEs
}

# Generate and save plots
plot_paths = []

for class_label in class_labels:
    class_name = class_label_mapping[str(class_label)]

    # Filter UE1 dataset for the given class
    class_df_ue1 = df_filtered[df_filtered['multiclass_label'] == class_label]

    # Filter Normal UEs using timestamps
    class_df_normal = df_normal_avg[df_normal_avg['_time'].isin(class_df_ue1['_time'])]

    for feature in features:
        plt.figure(figsize=(4, 4))

        # Plot Malicious UE (UE1) - Solid Red Line
        sns.lineplot(
            data=class_df_ue1,
            x='time_step',
            y=feature,
            linewidth=4,
            alpha=0.95,
            color=ue_colors["Malicious"],
            label="Malicious UE"
        )

        # Plot Averaged Normal UEs - Dashed Blue Line
        sns.lineplot(
            data=class_df_normal,
            x='time_step',
            y=feature,
            linewidth=4,
            alpha=0.95,
            color=ue_colors["Normal"],
            label="Averaged Normal UEs"
        )

        # Improve readability
        plt.xlabel('Time Step', fontsize=14)
        plt.ylabel(f'{feature} (Mbps)' if "bitrate" in feature else feature, fontsize=14)
        plt.xticks(fontsize=12)
        plt.yticks(fontsize=12)
        plt.legend(fontsize=12)

        plt.grid(True, linestyle="--", alpha=0.5)

        # Save as PDF
        file_name = f"{feature}_{class_name}_comparison.pdf".replace(" ", "_")
        file_path = os.path.join(output_folder, file_name)
        plt.savefig(file_path, format="pdf", bbox_inches='tight', dpi=400)
        plt.close()

        plot_paths.append(file_path)

# Provide the paths to the generated plots
plot_paths
